# amazon26: Business Entity Resolution on Kaggle

This notebook runs the full pipeline and writes **`<TEAM_NAME>_submission.zip`** to `/kaggle/working`. Download it from the **Output** tab.

**Before you click Run All:**

1. **Add the challenge data** (*Add Input → Upload → New Dataset*). Upload the organisers' `student_resource/` folder, or just its `dataset/` folder. It must contain `train/train_source1.tsv` … `test/test_source3.tsv`. Keep the dataset **private**.
   If `utils/validate_submission.py` is included, the notebook runs it.
2. **Get the code**, in one of two ways:
   - *Settings → Internet → On*, and the notebook clones `REPO_URL`. For a private repo, add a GitHub token as a Kaggle secret named `GITHUB_TOKEN` (*Add-ons → Secrets*).
   - Or upload this repository as another Kaggle dataset. The notebook uses any input folder that contains `src/cli.py`.
3. Set `TEAM_NAME` in the next cell.

**Accelerator:** the pipeline (scikit-learn, RapidFuzz, sparse TF-IDF) runs on CPU. A GPU session works but the GPU stays idle.

In [ ]:
# ---- Configuration -------------------------------------------------------
TEAM_NAME = "amazon26"   # your registered team name -> <TEAM_NAME>_submission.zip
REPO_URL = "https://github.com/MounishSenisetty/amazon26.git"
REPO_BRANCH = "main"
EXTRA_ARGS = []          # extra flags for `src.cli run`, e.g. ["--k-name", "20", "--k-addr", "15"]
RUN_EVALUATE = True      # also write validation_report.json (per-country scores) for docs/METHODOLOGY.md
METHODOLOGY_FILE = None  # path to a filled-in methodology .md; None = docs/METHODOLOGY.md from the repo

INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
SCRATCH = "/tmp/amazon26"

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

INPUT_ROOT, WORK_DIR, SCRATCH = Path(INPUT_ROOT), Path(WORK_DIR), Path(SCRATCH)
OUT_DIR, MODEL_DIR = WORK_DIR / "output", WORK_DIR / "artifacts"
SCRATCH.mkdir(parents=True, exist_ok=True)


def sh(cmd, cwd=None, check=True, capture=False):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not capture:
            print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {' '.join(map(str, cmd))}")
    return proc.returncode, "".join(lines)


def find_first(pattern):
    hits = sorted(INPUT_ROOT.rglob(pattern)) if INPUT_ROOT.exists() else []
    return hits[0] if hits else None

## 1. Locate the challenge data and the code

In [ ]:
first = find_first("train/train_source1.tsv")
if first is None:
    raise FileNotFoundError(f"No train/train_source1.tsv under {INPUT_ROOT}: add the challenge data as an input.")
DATA_DIR = first.parent.parent
missing = [f"{s}/{s}_source{n}.tsv" for s in ("train", "test") for n in (1, 2, 3)
           if not (DATA_DIR / s / f"{s}_source{n}.tsv").is_file()]
missing += [] if (DATA_DIR / "train" / "train_ground_truth.tsv").is_file() else ["train/train_ground_truth.tsv"]
if missing:
    raise FileNotFoundError(f"{DATA_DIR} is missing {missing}")
print("data:", DATA_DIR)

cli = find_first("src/cli.py")
if cli is not None:
    CODE_DIR = SCRATCH / "code"
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(cli.parent.parent, CODE_DIR, ignore=shutil.ignore_patterns("__pycache__", ".git", "dataset"))
    print("code: copied from input", cli.parent.parent)
else:
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # no secret: clone anonymously (public repo)
    CODE_DIR = SCRATCH / "repo"
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    rc, out = sh(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, CODE_DIR], check=False, capture=True)
    if rc != 0:
        raise RuntimeError("git clone failed. Turn Internet on, add a GITHUB_TOKEN secret for a private repo, "
                           "or upload the repo as a Kaggle dataset.\n" + out.replace(url, REPO_URL))
    print("code: cloned", REPO_URL, "@", REPO_BRANCH)
if not (CODE_DIR / "src" / "cli.py").is_file():
    raise FileNotFoundError(f"{CODE_DIR} has no src/cli.py; check REPO_BRANCH")

VALIDATOR = find_first("validate_submission.py")
print("organiser validator:", VALIDATOR or "not found (the pipeline's own rule checks will be used)")

## 2. Python environment

The notebook installs the exact versions in `requirements.txt` into a separate virtualenv, so the outputs match what reviewers reproduce from the zip. Without internet it falls back to Kaggle's preinstalled packages. The pipeline also works with pandas 2.x and scikit-learn ≥ 1.5, but the versions will differ from `requirements.txt`, so re-run with internet on before the final submission.

In [ ]:
VENV = SCRATCH / "venv"
PY = VENV / "bin" / "python"
ok = sh([sys.executable, "-m", "venv", VENV], check=False)[0] == 0
ok = ok and sh([PY, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                "-r", CODE_DIR / "requirements.txt"], check=False)[0] == 0
if not ok:
    print("\n!! Could not install pinned requirements; falling back to the preinstalled environment.")
    PY = Path(sys.executable)
    sh([PY, "-m", "pip", "install", "-q", "rapidfuzz"], check=False)
sh([PY, "-c", "import sys, pandas, numpy, sklearn, rapidfuzz; "
    "print('python', sys.version.split()[0], '| pandas', pandas.__version__, '| numpy', numpy.__version__, "
    "'| scikit-learn', sklearn.__version__, '| rapidfuzz', rapidfuzz.__version__)"])

## 3. Train, tune the threshold, predict on test

In [ ]:
shutil.rmtree(OUT_DIR, ignore_errors=True)
sh([PY, "-m", "src.cli", "run", "--data-dir", DATA_DIR, "--model-dir", MODEL_DIR, "--out-dir", OUT_DIR, *EXTRA_ARGS],
   cwd=CODE_DIR)

In [ ]:
if RUN_EVALUATE:
    _, report = sh([PY, "-m", "src.cli", "evaluate", "--data-dir", DATA_DIR, "--model-dir", MODEL_DIR],
                   cwd=CODE_DIR, capture=True)
    report = json.loads(report[report.index("{"):])
    (WORK_DIR / "validation_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))

## 4. Validate the submission files

In [ ]:
if VALIDATOR:
    sh([PY, VALIDATOR, "--matching", OUT_DIR / "matching_results.tsv",
        "--candidate", OUT_DIR / "candidate_pairs.tsv", "--test-dir", DATA_DIR / "test"])
else:
    sh([PY, "-m", "src.cli", "check", "--data-dir", DATA_DIR, "--out-dir", OUT_DIR], cwd=CODE_DIR)

## 5. Build the submission zip

In [ ]:
cmd = [PY, CODE_DIR / "scripts" / "make_submission.py", "--team", TEAM_NAME, "--out-dir", OUT_DIR, "--dest", WORK_DIR]
if METHODOLOGY_FILE:
    cmd += ["--methodology", METHODOLOGY_FILE]
sh(cmd)
ZIP_PATH = WORK_DIR / f"{TEAM_NAME}_submission.zip"

try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(ZIP_PATH, Path.cwd())))
except Exception:
    pass
print("Download from the Output tab:", ZIP_PATH)